In [1]:
import time
import statistics
from typing import List, Dict, Optional

# ==========================================
# 🔹 EXPERIMENT CONFIGURATION
# ==========================================

CORPUS = [
    "I love machine learning",
    "Good morning",
    "How are you?",
    "Artificial intelligence is transforming industries"
]

# Simulating the "Knowledge Base" of the model
# In a real LLM, this is the weights file. Here, it's a dictionary.
KNOWLEDGE_BASE = {
    "zero_shot_capabilities": {
        "I love machine learning": "J'aime l'apprentissage automatique",
        "Good morning": "Bonjour"
    },
    # Few-shot allows the model to "generalize" to new patterns
    "few_shot_capabilities": {
        "How are you?": "Comment ça va ?",
        "Artificial intelligence is transforming industries": "L'IA transforme les industries"
    }
}

# ==========================================
# 🔹 MOCK LLM CLASS
# ==========================================

class MockLLM:
    """
    Simulates a Large Language Model (LLM) API.
    """

    def generate(self, prompt: str, examples: Optional[List[Dict[str, str]]] = None) -> str:
        """
        Simulates generating text based on a prompt and optional few-shot examples.

        Args:
            prompt (str): The input text to translate.
            examples (list): A list of example input/outputs (Few-Shot).

        Returns:
            str: The generated translation.
        """
        # 1. Simulate Inference Latency (Network + Computation)
        # Real-world latency is usually 200ms - 2s depending on model size.
        time.sleep(0.1)

        # 2. Logic: Zero-Shot vs Few-Shot
        if not examples:
            # --- ZERO-SHOT MODE ---
            # Model relies strictly on rigid pre-trained knowledge.
            return KNOWLEDGE_BASE["zero_shot_capabilities"].get(
                prompt,
                "❌ [Error] Model lacks context to translate."
            )
        else:
            # --- FEW-SHOT MODE ---
            # The model uses the examples to "understand" the task better.
            # We simulate this by unlocking the "few_shot_capabilities" dictionary.

            # First, check basic knowledge
            if prompt in KNOWLEDGE_BASE["zero_shot_capabilities"]:
                return KNOWLEDGE_BASE["zero_shot_capabilities"][prompt]

            # Then, check extended knowledge (simulating generalization)
            if prompt in KNOWLEDGE_BASE["few_shot_capabilities"]:
                return KNOWLEDGE_BASE["few_shot_capabilities"][prompt]

            return "❌ [Error] Even with examples, translation failed."

# ==========================================
# 🔹 EVALUATION ENGINE
# ==========================================

def run_evaluation(model: MockLLM, dataset: List[str], strategy_name: str, examples: list = None):
    """
    Runs the benchmark for a specific prompting strategy.
    """
    print(f"\n{'='*20} {strategy_name.upper()} EVALUATION {'='*20}")

    latencies = []

    if examples:
        print(f"ℹ️  Context provided: {len(examples)} examples injected into prompt.\n")
    else:
        print(f"ℹ️  Context provided: None (Zero-Shot).\n")

    print(f"{'INPUT':<50} | {'OUTPUT':<40} | {'LATENCY'}")
    print("-" * 105)

    for sentence in dataset:
        # --- START TIMER ---
        start_time = time.time()

        # Call the model
        response = model.generate(sentence, examples=examples)

        # --- STOP TIMER ---
        end_time = time.time()

        latency = end_time - start_time
        latencies.append(latency)

        # Formatting output
        print(f"{sentence:<50} | {response:<40} | {latency:.4f}s")

    # --- METRICS CALCULATION ---
    avg_latency = statistics.mean(latencies)
    print("-" * 105)
    print(f"📊 {strategy_name} Average Latency: {avg_latency:.4f}s")

# ==========================================
# 🔹 MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    # Initialize our mock AI
    my_model = MockLLM()

    # 1. Run Zero-Shot (No examples)
    run_evaluation(
        model=my_model,
        dataset=CORPUS,
        strategy_name="Zero-Shot",
        examples=None
    )

    # 2. Define Few-Shot Examples (The "Context")
    few_shot_context = [
        {"input": "Hello", "output": "Bonjour"},
        {"input": "Thank you", "output": "Merci"}
    ]

    # 3. Run Few-Shot (With examples)
    run_evaluation(
        model=my_model,
        dataset=CORPUS,
        strategy_name="Few-Shot",
        examples=few_shot_context
    )

    # 4. Final Industry Conclusion
    print("\n================ 🏆 EXPERIMENT CONCLUSION ================")
    print("1. Accuracy: Few-shot prompting significantly improved model coverage on complex inputs.")
    print("2. Latency:  Both methods had similar latency in this simulation (0.1s).")
    print("             *Note: In real production, Few-shot is slightly slower due to processing more input tokens.*")
    print("3. Recommendation: Use Few-shot prompting for tasks requiring specific formatting or tone.")


==================== ZERO-SHOT EVALUATION ====================
ℹ️  Context provided: None (Zero-Shot).

INPUT                                              | OUTPUT                                   | LATENCY
---------------------------------------------------------------------------------------------------------
I love machine learning                            | J'aime l'apprentissage automatique       | 0.1003s
Good morning                                       | Bonjour                                  | 0.1010s
How are you?                                       | ❌ [Error] Model lacks context to translate. | 0.1001s
Artificial intelligence is transforming industries | ❌ [Error] Model lacks context to translate. | 0.1002s
---------------------------------------------------------------------------------------------------------
📊 Zero-Shot Average Latency: 0.1004s

==================== FEW-SHOT EVALUATION ====================
ℹ️  Context provided: 2 examples injected into prompt.

I